# Лекция 14. FastAPI, Docker и поставка приложения

Веб-API — это обычные предметные функции, к которым добавлены HTTP-маршруты, проверка входа и преобразование ответа. Container — запущенный процесс из заранее собранного image. В этой лекции пройдём весь короткий путь: функция добавляет задачу, FastAPI делает её доступной по HTTP, тест вызывает приложение без настоящего сервера, Dockerfile собирает image, а CI повторяет проверку на чистой машине.

## Цели

После лекции вы сможете:

- спроектировать небольшой REST API вокруг ресурсов;
- различать path, query и body;
- описывать вход и выход Pydantic-моделями;
- возвращать подходящие статусы и ошибки;
- внедрять репозиторий через `Depends`;
- управлять общим ресурсом через `lifespan`;
- тестировать API через `TestClient`;
- различать Docker image, container и volume;
- объяснить каждую строку минимального Dockerfile и Compose-файла;
- передавать конфигурацию через окружение;
- добавить healthcheck и CI;
- назвать, чего не хватает между `docker run` и production-деплоем.

## Перед началом

Нужны HTTP из занятия 12, классы и зависимости из занятия 10, базы из занятия 13 и структура проекта из занятия 4. Целевая версия — Python 3.14. На FastAPI и Pydantic заложено около 30 минут, на зависимости, lifespan и тест — 15 минут, на Docker и Compose — 25 минут, на конфигурацию, healthcheck и CI — 10 минут, на поставку, неожиданные случаи и вопросы — 10 минут.

Рядом с ноутбуком находится [полный минимальный пример](example/). Он хранит данные в памяти: это позволяет увидеть поставку приложения, не повторяя всю лекцию о базе.

## Что происходит с запросом

1. ASGI-сервер принимает HTTP-запрос.
2. FastAPI выбирает маршрут по методу и пути.
3. Path/query/header/body преобразуются и проверяются.
4. Система зависимостей получает нужные ресурсы.
5. Вызывается наша функция маршрута.
6. Предметное ядро и репозиторий выполняют работу.
7. Модель ответа фильтрует и сериализует результат.
8. Сервер отправляет статус, заголовки и JSON.

Фреймворк автоматизирует повторяющуюся обвязку. Решение, можно ли завершить задачу или кто имеет право это сделать, остаётся предметным кодом.

## REST: проектируем ресурсы, а не команды Python

Для списка задач естественны адреса:

- `GET /tasks` — получить коллекцию;
- `POST /tasks` — создать элемент;
- `GET /tasks/{task_id}` — получить элемент;
- `PATCH /tasks/{task_id}` — частично изменить;
- `DELETE /tasks/{task_id}` — удалить.

REST — набор архитектурных ограничений, а не автоматическая гарантия хорошего API. В минимальном примере действие завершения записано как `POST /tasks/{id}/done`: это понятный прикладной endpoint, хотя универсальное API могло бы использовать `PATCH`. Важнее стабильный контракт и ясный смысл, чем механическое поклонение URL.

## Path, query и body

- Path определяет конкретный ресурс: `/tasks/42`.
- Query уточняет чтение: `/tasks?done=false&limit=20`.
- Body передаёт структурированные данные создания или изменения.
- Header несёт метаданные: авторизацию, формат, идентификатор запроса.

FastAPI выводит источник из объявления функции: имя, присутствующее в шаблоне пути, становится path-параметром; простой тип обычно query-параметром; Pydantic-модель — телом JSON. Для сложного случая источник задают явно через `Path`, `Query`, `Header` или `Body`.

## Pydantic: контракт входа и выхода

Модель входа содержит только то, что клиенту разрешено прислать. Серверный `id` и `done` в неё не входят. Модель ответа содержит только публичные поля. Если объект хранения имеет внутренний токен или хеш пароля, `response_model` не должен его пропускать.

Проверка типов не заменяет предметные правила, но отсекает пустой title, приоритет вне диапазона и неверную форму JSON до вызова маршрута.

In [ ]:
from pydantic import BaseModel, Field, field_validator

class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=120)
    priority: int = Field(default=3, ge=1, le=5)

    @field_validator("title")
    @classmethod
    def normalize_title(cls, value: str) -> str:
        normalized = value.strip()
        if not normalized:
            raise ValueError("title must not be blank")
        return normalized

class TaskPublic(BaseModel):
    id: int
    title: str
    priority: int
    done: bool

assert TaskCreate(title="  Prepare demo  ").title == "Prepare demo"

> **Современный Pydantic v2.** Для превращения модели в словарь используется `model_dump()`, для копии с изменениями — `model_copy()`, для проверки другого объекта — `model_validate()`. В старом коде Pydantic v1 часто встречаются `.dict()`, `.copy()` и `.parse_obj()`. FastAPI поддерживал период миграции, но материалы курса используют актуальный API v2.

Некорректное тело обычно получает `422 Unprocessable Content`, и функция маршрута не запускается. Успешное создание ресурса возвращает `201 Created`, чтение — `200 OK`, удаление без тела может вернуть `204 No Content`. Отсутствующий ресурс — `404`, конфликт текущего состояния — `409`, отсутствие аутентификации — `401`, недостаток прав — `403`.

Статус выбирается по смыслу HTTP-контракта. Возвращать `200` с телом `{"error": ...}` неудобно: клиенты, прокси и мониторинг видят успех.

## Репозиторий отделяет HTTP от хранения

Маршрут не должен знать, лежат задачи в словаре, SQLite или внешнем API. Узкий `Protocol` описывает нужные операции. Учебный `MemoryTaskRepository` хранит объекты в словаре; позже его можно заменить реализацией над базой, не меняя контракт маршрута.

Это не обязательный слой для каждой функции. Он полезен здесь, потому что хранение действительно меняется и его нужно подменять в тесте.

In [ ]:
from typing import Protocol

class TaskRepository(Protocol):
    def add(self, data: TaskCreate) -> TaskPublic: ...
    def all(self) -> list[TaskPublic]: ...
    def mark_done(self, task_id: int) -> TaskPublic | None: ...

class MemoryTaskRepository:
    def __init__(self) -> None:
        self.tasks: dict[int, TaskPublic] = {}
        self.next_id = 1

    def add(self, data: TaskCreate) -> TaskPublic:
        task = TaskPublic(id=self.next_id, title=data.title, priority=data.priority, done=False)
        self.tasks[task.id] = task
        self.next_id += 1
        return task

    def all(self) -> list[TaskPublic]:
        return sorted(self.tasks.values(), key=lambda task: (-task.priority, task.id))

    def mark_done(self, task_id: int) -> TaskPublic | None:
        task = self.tasks.get(task_id)
        if task is None:
            return None
        updated = task.model_copy(update={"done": True})
        self.tasks[task_id] = updated
        return updated

## Маршрут — обычная функция с декоратором

Декоратор связывает метод и путь с функцией. `response_model` документирует и фильтрует выход, `status_code` задаёт успешный статус. `HTTPException` немедленно формирует известную HTTP-ошибку. Не следует использовать её глубоко в предметном ядре: ядро поднимает предметную ошибку, а HTTP-слой переводит её в статус.

In [ ]:
from fastapi import FastAPI, HTTPException, status

repository = MemoryTaskRepository()
app = FastAPI(title="Course Task API")

@app.post("/tasks", response_model=TaskPublic, status_code=status.HTTP_201_CREATED)
def create_task(data: TaskCreate) -> TaskPublic:
    return repository.add(data)

@app.get("/tasks", response_model=list[TaskPublic])
def list_tasks() -> list[TaskPublic]:
    return repository.all()

@app.post("/tasks/{task_id}/done", response_model=TaskPublic)
def mark_done(task_id: int) -> TaskPublic:
    task = repository.mark_done(task_id)
    if task is None:
        raise HTTPException(status_code=404, detail="Task not found")
    return task

Из типов и маршрутов FastAPI строит OpenAPI-схему и интерфейсы `/docs` и `/redoc`. Это полезный способ исследовать API и договориться с клиентом. Но интерактивная документация не заменяет тест: человек нажал кнопку один раз, а тест фиксирует ожидаемый контракт и повторяется после каждого изменения.

## Dependency injection без магического контейнера

Dependency — функция, которую FastAPI вызывает перед маршрутом и чей результат передаёт параметру. Она может получить session базы, настройки, текущего пользователя или репозиторий. Зависимость с `yield` освобождает ресурс после ответа.

Современный стиль использует `Annotated[T, Depends(provider)]`: основной тип остаётся виден редактору, а метаданные объясняют FastAPI способ получения значения.

In [ ]:
from typing import Annotated
from fastapi import Depends, Request

def get_repository(request: Request) -> TaskRepository:
    return request.app.state.repository

RepositoryDep = Annotated[TaskRepository, Depends(get_repository)]

# В реальном app функция маршрута объявляет зависимость так:
# def list_tasks(tasks: RepositoryDep) -> list[TaskPublic]:
#     return tasks.all()

## `lifespan`: общий ресурс живёт вместе с приложением

Engine базы, пул клиента или большая модель создаются один раз перед приёмом запросов и освобождаются после остановки. Импорт модуля не должен самопроизвольно открывать production-соединение или скачивать модель.

> **Актуальная рекомендация FastAPI.** Используйте параметр `lifespan` и асинхронный контекстный менеджер. Старые обработчики `@app.on_event('startup')` и `shutdown` считаются альтернативным устаревшим интерфейсом; смешивать оба механизма в одном приложении не следует.

In [ ]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.repository = MemoryTaskRepository()
    yield
    # Здесь закрывают pool/client; словарю специальная очистка не нужна.

app_with_lifespan = FastAPI(lifespan=lifespan)

## `def` или `async def`

Если используемая библиотека синхронная и блокирует поток, обычный `def`-маршрут честно сообщает это FastAPI, и фреймворк может выполнить его в thread pool. Если библиотека предоставляет `await`, маршрут делают `async def`.

Нельзя вызывать `requests.get()`, `time.sleep()` или тяжёлый SQL через синхронный драйвер прямо внутри `async def`: это блокирует event loop. И наоборот, `async def` вокруг чистого короткого вычисления не приносит ускорения. Выбор следует из реального I/O, изученного в занятии 12.

## Тестируем приложение без сетевого порта

`TestClient` посылает запрос приложению внутри процесса. Контекст `with TestClient(app)` запускает и завершает lifespan. Вход проверяется тем же Pydantic, маршрутизация и dependency injection работают по-настоящему, но поднимать Uvicorn и искать свободный порт не нужно.

Для внешнего API или базы зависимость подменяют через `app.dependency_overrides`. После теста словарь overrides очищают, чтобы тесты не влияли друг на друга. Фабрика `create_app()` ещё проще создаёт независимый экземпляр с тестовым репозиторием.

In [ ]:
from fastapi.testclient import TestClient

test_app = FastAPI()
test_repository = MemoryTaskRepository()

@test_app.post("/tasks", response_model=TaskPublic, status_code=201)
def test_create_route(data: TaskCreate) -> TaskPublic:
    return test_repository.add(data)

client = TestClient(test_app)
response = client.post("/tasks", json={"title": "Prepare demo", "priority": 5})
assert response.status_code == 201
assert response.json()["title"] == "Prepare demo"
assert client.post("/tasks", json={"title": "", "priority": 9}).status_code == 422

## От запуска к поставке

Локально разработчик знает, какой Python установить, где лежит код и какую команду выполнить. Поставка должна сделать эти предположения явными и воспроизводимыми:

- версия runtime и системная среда;
- зависимости;
- исходный код;
- команда запуска;
- конфигурация;
- внешние ресурсы;
- проверка здоровья;
- повторяемая автоматическая проверка.

Docker решает упаковку процесса и файлов. Он не выдаёт домен, TLS-сертификат, резервную копию базы и дежурного инженера.

## Image и container

**Image** — неизменяемый шаблон со слоями файлов и метаданными запуска. **Container** — запущенный экземпляр image с процессом и собственным записываемым слоем. Из одного image можно запустить несколько containers.

Контейнер существует, пока работает его главный процесс. Если процесс завершился, контейнер остановился. Это не виртуальная машина с обязательным init и множеством сервисов; для нашего приложения главным процессом является FastAPI/Uvicorn.

## Минимальный Dockerfile

Файл [example/Dockerfile](example/Dockerfile) начинается с официального Python 3.14 image, задаёт каталог, сначала копирует зависимости для кэша, устанавливает их, затем копирует часто меняющийся код. `CMD` записан в exec-форме списка: тогда сервер получает сигналы остановки напрямую и корректно выполняет lifespan shutdown.

In [ ]:
dockerfile = '''FROM python:3.14-slim
WORKDIR /code
COPY requirements.txt /code/requirements.txt
RUN python -m pip install --no-cache-dir -r /code/requirements.txt
COPY app /code/app
EXPOSE 8000
CMD ["fastapi", "run", "app/main.py", "--port", "8000"]
'''
assert "python:3.14-slim" in dockerfile
assert 'CMD ["fastapi"' in dockerfile

`docker build -t course-task-api .` передаёт текущий каталог как build context. Точка в конце — аргумент, а не украшение. `.dockerignore` не отправляет `.git`, `.venv`, кэши, тестовые артефакты и секретные локальные файлы.

Каждая инструкция образует кэшируемый слой. Если сначала выполнить `COPY . .`, изменение одной строки приложения инвалидирует слой и заставит заново устанавливать зависимости. Поэтому файл зависимостей копируют раньше кода. Кэш ускоряет сборку, но не должен скрывать незакреплённые или невоспроизводимые зависимости.

## Минимальный image и production-hardening

Учебный Dockerfile намеренно короткий и по умолчанию запускает процесс от root внутри container. Для реальной поставки дополнительно рассматривают непривилегированного пользователя, точную фиксацию зависимостей, регулярную пересборку на обновлённом base image, сканирование уязвимостей, read-only filesystem там, где он возможен, и лимиты ресурсов.

Нельзя скопировать случайный hardening-чек-лист и считать безопасность законченной. Например, read-only root filesystem потребует явно выделить доступные для записи каталоги, а непривилегированный пользователь должен иметь права на нужные файлы. Каждая мера проверяется запуском приложения и моделью угроз.

## Порт внутри и снаружи

`EXPOSE 8000` документирует, что процесс слушает порт 8000 внутри контейнера, но не публикует его на host. Команда `docker run -p 8080:8000 image` связывает host-порт 8080 с container-портом 8000.

Внутри контейнера `localhost` означает сам контейнер. Если API ищет базу в другом Compose-сервисе, адресом будет имя сервиса вроде `db`, а не `localhost`. Снаружи пользователь обращается к опубликованному host-порту.

## Конфигурация и секреты

Адрес базы, режим логирования и имя приложения меняются между окружениями, поэтому передаются через environment или систему конфигурации. Код задаёт безопасные значения по умолчанию только там, где они действительно безопасны.

Пароль, API-токен и приватный ключ нельзя записывать в Dockerfile через `ENV`, копировать в image или коммитить в Compose. Даже удалённый на следующем слое файл мог остаться в истории image. Production-секреты поставляет платформа или secret manager во время запуска, а логи не печатают их.

## Compose описывает совместный запуск

[example/compose.yaml](example/compose.yaml) задаёт сервис, build context, публикацию порта, environment и healthcheck. Команды пишутся `docker compose up --build`, `docker compose ps`, `docker compose down`.

> **Современный Compose.** Верхнее поле `version: '3.8'` устарело и оставлено только для обратной совместимости. Compose использует текущую Specification, поэтому новый файл начинается с `services:`. Это не версия самого установленного Docker Compose.

In [ ]:
compose_fragment = '''services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      APP_NAME: "Course Task API"
'''
assert compose_fragment.startswith("services:")
assert "version:" not in compose_fragment

## Healthcheck: какой вопрос мы задаём

Liveness отвечает: «процесс не застрял безнадёжно?» Readiness: «экземпляр готов принимать запросы прямо сейчас?» Иногда это разные endpoints. Проверка `/health` с постоянным `200` доказывает только работу маршрута. Readiness может проверить обязательную базу, но не должна падать из-за необязательной аналитики и не должна создавать большую нагрузку.

Compose отмечает container как healthy по команде healthcheck. Для порядка запуска другого сервиса простого `depends_on` недостаточно: ожидание готовности требует условия `service_healthy` и healthcheck зависимости.

## Файлы контейнера не являются хранилищем

Записи в writable layer принадлежат конкретному container и исчезают при его удалении. Image остаётся прежним. Для постоянных данных используют внешнюю базу или volume; для разработки код можно подключить bind mount.

Volume не заменяет резервное копирование. Если SQLite-файл хранится в volume, данные переживут пересоздание container, но всё ещё требуют согласованного backup, проверки восстановления и осторожности с несколькими процессами записи.

## CI повторяет проверку на чистой машине

Continuous Integration запускается при push и pull request: получает репозиторий, ставит Python 3.14 и зависимости, выполняет Ruff и pytest. При необходимости следующим шагом собирается Docker image. CI должен воспроизводить команды README, а не иметь тайный второй способ установки.

> **Актуально на момент курса.** Официальный пример GitHub Actions использует `actions/checkout@v6` и `actions/setup-python@v5`. Номера сторонних actions со временем обновляются; Dependabot или регулярная ревизия не дают инфраструктурному коду застыть навсегда.

In [ ]:
ci_steps = [
    "actions/checkout@v6",
    "actions/setup-python@v5: python 3.14",
    "python -m pip install -r requirements-dev.txt",
    "ruff check .",
    "pytest",
]
assert ci_steps[-2:] == ["ruff check .", "pytest"]

## `docker run` ещё не production-деплой

Чтобы приложение было доступно пользователям, нужны решения для:

- домена, DNS и HTTPS;
- запуска после перезагрузки и restart policy;
- числа процессов или replicas;
- применения миграций один раз, а не каждым worker;
- внешней базы и резервных копий;
- секретов и прав доступа;
- структурированных логов, метрик и alerting;
- лимитов CPU/памяти и graceful shutdown;
- публикации новой версии и rollback.

На одном сервере это может дать Docker Compose и reverse proxy, в облаке — управляемая платформа. Kubernetes не является обязательной следующей ступенью маленького учебного проекта.

Несколько workers — отдельные процессы с отдельной памятью. Наш `MemoryTaskRepository` в каждом worker будет своим: созданная задача может «пропасть» при следующем запросе к другому процессу. Это нормальное следствие модели процессов и ещё одна причина хранить общее состояние во внешней базе.

Число workers не выбирают по формуле из воздуха: учитывают CPU, память, I/O и измерения. В оркестраторе часто запускают один процесс на container и масштабируют containers; на простом сервере несколько Uvicorn workers внутри одного container могут быть разумны.

## Неожиданно, но по правилам

### 1. Аннотация типа реально меняет HTTP-поведение

FastAPI читает её во время выполнения. `task_id: int` превращает строку пути в число или возвращает `422` до вызова функции. Это больше, чем подсказка статического анализатора.

### 2. `async def` может заблокировать приложение

Синхронный запрос или `time.sleep` внутри него занимает event loop. Обычный `def` с блокирующей библиотекой иногда корректнее.

### 3. Модель ответа может скрыть лишнее поле

Даже если endpoint вернул объект с `password_hash`, `response_model` без этого поля отфильтрует выход. Но полагаться только на это без теста утечки нельзя.

### 4. `EXPOSE` не открывает порт пользователю

Нужен `docker run -p host:container` или поле `ports` Compose. `EXPOSE` — метаданные image.

### 5. `localhost` меняет смысл внутри container

Он указывает на тот же container, а не на ноутбук и не на соседний сервис.

### 6. Записанный в container SQLite-файл исчезает вместе с container

Перезапуск того же экземпляра и удаление с созданием нового — разные операции. Для данных нужен volume или внешняя БД.

### 7. `depends_on` не всегда означает «база готова»

Запущенный процесс базы может ещё выполнять восстановление. Нужны healthcheck и условие готовности либо retry клиента.

### 8. Удаление секрета следующей строкой Dockerfile не стирает прошлый слой

Секрет мог сохраниться в истории image и build cache. Его вообще не копируют в build context.

### 9. Зелёный `/health` не доказывает полезность приложения

Маршрут может отвечать из памяти, пока база недоступна. Healthcheck должен отвечать на конкретный вопрос liveness или readiness.

### 10. «Работает в Docker» не означает «развёрнуто»

Image решает упаковку. Доступность, TLS, миграции, backup, наблюдаемость и rollback остаются отдельной инженерной работой.

## Самопроверка

1. Какие этапы проходит HTTP-запрос в FastAPI?
2. Чем path отличается от query и body?
3. Зачем разделять модели входа и ответа?
4. Что изменилось в API Pydantic v2?
5. Почему ошибка валидации возникает до маршрута?
6. Когда возвращать `404`, а когда `409`?
7. Что делает dependency injection?
8. Зачем ресурсу `lifespan`?
9. Когда маршрут писать через `def`, а когда через `async def`?
10. Что проверяет `TestClient`?
11. Чем image отличается от container?
12. Почему зависимости копируют раньше кода?
13. Что делает `.dockerignore`?
14. Почему `EXPOSE` недостаточно?
15. Где хранить конфигурацию и секреты?
16. Чем health readiness отличается от liveness?
17. Почему нужен volume?
18. Что запускает CI?
19. Почему in-memory repository ломается с несколькими workers?
20. Чего Docker не решает в production?

## Источники

- [FastAPI: request body](https://fastapi.tiangolo.com/tutorial/body/), [response model](https://fastapi.tiangolo.com/tutorial/response-model/) и [dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/).
- [FastAPI lifespan](https://fastapi.tiangolo.com/advanced/events/) и [тестирование](https://fastapi.tiangolo.com/tutorial/testing/).
- [FastAPI в контейнерах](https://fastapi.tiangolo.com/deployment/docker/) — актуальный Dockerfile с Python 3.14.
- [Dockerfile reference](https://docs.docker.com/reference/dockerfile) — слои, exec form, `EXPOSE`, `HEALTHCHECK`.
- [Compose Specification](https://docs.docker.com/compose/compose-file/) и [healthcheck](https://docs.docker.com/reference/compose-file/services/#healthcheck).
- [Docker storage](https://docs.docker.com/engine/storage/) — writable layer и volumes.
- [GitHub Actions: Python](https://docs.github.com/en/actions/tutorials/build-and-test-code/python) — текущие checkout/setup-python и CI-шаги.

Версии внешних actions и фреймворков проверяются при обновлении курса; номера не являются частью языка Python.

## Итоги

- FastAPI связывает HTTP-контракт с обычными Python-функциями и типами.
- Pydantic проверяет вход, а модель ответа фильтрует публичный результат.
- Dependencies передают ресурсы, lifespan управляет их общим временем жизни.
- `TestClient` проверяет маршрутизацию и контракт без запуска сетевого сервера.
- Docker image упаковывает runtime, зависимости, код и команду; container запускает процесс из image.
- Compose описывает совместный запуск, environment и healthcheck.
- Постоянные данные не хранят в writable layer container.
- CI повторяет lint и тесты на чистой машине.
- Поставка требует также TLS, миграций, секретов, backup, наблюдаемости и rollback.
- Ни FastAPI, ни Docker не обязательны для проекта: технология оправдана только задачей.

На семинаре команды показывают законченный пользовательский сценарий и защищают свои решения.